# Location selection

In [141]:
from desdeo.problem import Constant, Variable, Problem, Objective, VariableTypeEnum
import numpy as np

# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Model inputs

In [142]:
# Mininum expected attendance to be worth visiting 
min_att = 15
raw_max_events = 8
raw_dollars_per_gallon = 3.00
raw_mpg = 6.0

## Load and process constants


In [143]:
from slugify import slugify
import pandas as pd

def no_nan(val):
    if pd.isna(val):
        return ""
    else:
        return str(val)

home = "Ada"
# Read the adjacency matrix 
adjacencies = pd.read_csv("adjacencyMatrix.csv", index_col=0)
dist2home = adjacencies.loc[:,[home]].rename(columns={home:"dist2home"})
dist2home.index.name = "city"

# Read the cities 
cities = pd.read_csv("cities.csv")
sites = pd.read_csv("sites.csv")

# Create event table
events = pd.merge(cities,sites,on="city")
events.loc[:,"expectedAttendance"] = (events.loc[:,"pop"] * events.loc[:,"attendanceRate"]).astype(int)

events.loc[:, "event_id"] = events.apply(lambda row: slugify(f'{row["city"]} {row["site"]} {no_nan(row["event"])}'), axis=1)

# Add the distance to home for each event 
events = pd.merge(events, dist2home, on="city")

events

,city,lat,long,pop,site,event,attendanceRate,expectedAttendance,event_id,dist2home
0,Ada,40.768056,-83.825278,5334,Public Library,NaN,0.002,10,ada-public-library,0.0
1,Lima,40.746389,-84.123333,35579,Mercy Health Thrift,NaN,0.002,71,lima-mercy-health-thrift,16.2
2,Lima,40.746389,-84.123333,35579,Habitat For Humanity,NaN,0.002,71,lima-habitat-for-humanity,16.2
3,Lima,40.746389,-84.123333,35579,Our Daily Bread,NaN,0.002,71,lima-our-daily-bread,16.2
4,Lima,40.746389,-84.123333,35579,St. Mark’s Methodist,Community Meal,0.002,71,lima-st-marks-methodist-community-meal,16.2
5,Lima,40.746389,-84.123333,35579,Christian Corner Community Center,NaN,0.002,71,lima-christian-corner-community-center,16.2
6,Kenton,40.646667,-83.622500,7947,Seton Hall,NaN,0.002,15,kenton-seton-hall,15.5
7,Kenton,40.646667,-83.622500,7947,Hardincrest,NaN,0.002,15,kenton-hardincrest,15.5
8,Kenton,40.646667,-83.622500,7947,YMCA,NaN,0.002,15,kenton-ymca,15.5
9,Delphos,40.861111,-84.350000,7117,Public Library,NaN,0.002,14,delphos-public-library,31.8


# Constants

In [157]:
event_count = events.shape[0]

# Expected attenance 
raw_attendance = events.loc[:,"expectedAttendance"] 
raw_over_attendance = raw_attendance < min_att

# Expected attendance 
exp_att = []
# Over staffed events (ose)
ose = []
for e in range(event_count): 
    exp_att.append(Constant(name="Expected attendance", 
                        symbol=f"exp_att_{e}", 
                        type="int", 
                        shape=[events.shape[0]],
                        value=raw_attendance[e]
                        ))

    ose.append(Constant(name="Over attendance", 
                        symbol=f"ose_{e}", 
                        type="binary",
                        shape=[events.shape[0]], 
                        value=raw_over_attendance[e]))

mpg = Constant(name="Miles per gallon", 
               symbol="mpg", 
               type="real",
               value=raw_mpg)

dpg = Constant(name="Dollars per gallon", 
               symbol="dpg", 
               type="real",
               value=raw_dollars_per_gallon)


max_events = Constant(name="Maximum events",
                      symbol="max_evt", 
                      type="int", 
                      value=raw_max_events)






## Variables
Generate one variable per event

In [158]:

#evt_visited = TensorVariable(
#        name="Event visited",
#        symbol=f"evt_visited",
#        shape=[event_count],
#        variable_type="binary",
#        lowerbounds=[0] * event_count,
#        upperbounds=[1] * event_count,
#        initial_values=[0] * event_count)

evt_visited = []
for e in range(event_count):
  evt_visited.append(Variable(
    name=events.loc[e, "event_id"],
    symbol=f"evt_visit_{e}",
    variable_type="binary",
    lowerbound=0,
    upperbound=1,
    initial_value=0))

#vars = []
#
#for event_no in range(event_count):
#    vars.append(Variable(
#        name=events.loc[event_no, "event_id"],
#        symbol=f"x_{event_no}",
#        variable_type=VariableTypeEnum.binary,
#        lowerbound=0,
#        upperbound=1,
#        initial_value=0))




## Objectives

### Build the function strings

In [159]:
# total patients served
obj_func_total_patients = " + ".join([f"evt_visit_{e} * exp_att_{e}" for e in range(event_count)])
obj_func_total_patients = f"Sum({obj_func_total_patients})" 
display(obj_func_total_patients)


# ose = over staffed events
obj_func_ose = " + ".join([f"evt_visit_{e} * ose_{e}" for e in range(event_count)])
obj_func_ose = f"Sum({obj_func_ose})" 
obj_func_ose 



'Sum(evt_visit_0 * exp_att_0 + evt_visit_1 * exp_att_1 + evt_visit_2 * exp_att_2 + evt_visit_3 * exp_att_3 + evt_visit_4 * exp_att_4 + evt_visit_5 * exp_att_5 + evt_visit_6 * exp_att_6 + evt_visit_7 * exp_att_7 + evt_visit_8 * exp_att_8 + evt_visit_9 * exp_att_9 + evt_visit_10 * exp_att_10 + evt_visit_11 * exp_att_11 + evt_visit_12 * exp_att_12 + evt_visit_13 * exp_att_13 + evt_visit_14 * exp_att_14 + evt_visit_15 * exp_att_15 + evt_visit_16 * exp_att_16 + evt_visit_17 * exp_att_17)'

'Sum(evt_visit_0 * ose_0 + evt_visit_1 * ose_1 + evt_visit_2 * ose_2 + evt_visit_3 * ose_3 + evt_visit_4 * ose_4 + evt_visit_5 * ose_5 + evt_visit_6 * ose_6 + evt_visit_7 * ose_7 + evt_visit_8 * ose_8 + evt_visit_9 * ose_9 + evt_visit_10 * ose_10 + evt_visit_11 * ose_11 + evt_visit_12 * ose_12 + evt_visit_13 * ose_13 + evt_visit_14 * ose_14 + evt_visit_15 * ose_15 + evt_visit_16 * ose_16 + evt_visit_17 * ose_17)'

### Create the objective objects

In [160]:

# Construct the total patients visited objective function


# Create objects
total_patients = Objective(
    name = "Maximize total patients visited",
    symbol = "f_1", 
    maximize = True,
    func = obj_func_total_patients
)

over_staffed_events = Objective(
    name = "Minimize the number of overstaffed events",
    symbol = "f_2",
    maximize = False,
    func = obj_func_ose
)

### Objective 4: Minimize costs 
## How many miles are driven/gas costs
#total_distance = np.apply_along_axis(lambda event_mask: self.events.loc[event_mask, "dist2home"].sum(), 1, x_bool) * self.gas_price * 2
#total_gas_cost = (total_distance / self.mpg) * self.gas_price
#
## How much are we paying the driver per trip 
#total_personel_paid = x_bool.sum(axis=1) * self.driver_fee
#
## TODO driver fee
#total_travel_costs = total_gas_cost + total_personel_paid

#total_costs = Objective(
#    name = "Minimize costs",
#    symbol = "f_3",
#    maximize=False,
#    func = ""
#)

## Problem 


In [161]:


prob = Problem(
        name="Simple site selection",
        description="Simple implementation of the site selection problem",
        type="linear",
        constants=exp_att +  ose + [mpg, dpg, max_events],
        variables=evt_visited,
        objectives=[total_patients, over_staffed_events],
        constraints=[]
    )


## Ideal/nadir


In [165]:
# f_1 ideal is seeing all patients, nadir is seeing no patients
all_patients = int(np.sum(events.loc[:,"expectedAttendance"]))

# f_2 ideal is having no over staffed events
# f_2 naird is having visiting all locations with over staffed events
all_ose = int(np.sum(raw_over_attendance))

prob = prob.update_ideal_and_nadir(
    new_ideal={
        "f_1": all_patients,
        "f_2": 0
        }, 
    new_nadir={
        "f_1": 0,
        "f_2": all_ose
        }
    )
print(f"Ideal values: {prob.get_ideal_point()}")
print(f"Nadir values: {prob.get_nadir_point()}")

Ideal values: {'f_1': 441, 'f_2': 0}
Nadir values: {'f_1': 0, 'f_2': 10}


## Solve!

In [162]:
# This raised a "SymPy evaluator does not yet support tensors."
#from desdeo.mcdm.reference_point_method import rpm_solve_solutions
#
#reference_point = {"f_1": all_patients}
#
#results = rpm_solve_solutions(prob, reference_point=reference_point)

#from desdeo.tools import add_asf_diff
#
#reference_point = {"f_1": 20}
#prob_w_asf = add_asf_diff(prob, symbol="asf", reference_point=reference_point)

from desdeo.emo.methods.EAs import nsga3_mixed_integer



solver, publisher = nsga3_mixed_integer(problem=prob)

result = solver()



In [163]:
result

EMOResult(solutions=shape: (500, 18)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ evt_visit ┆ evt_visit ┆ evt_visit ┆ evt_visit ┆ … ┆ evt_visit ┆ evt_visit ┆ evt_visit ┆ evt_visi │
│ _0        ┆ _1        ┆ _2        ┆ _3        ┆   ┆ _14       ┆ _15       ┆ _16       ┆ t_17     │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ f64       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1.0       ┆ 1.0       ┆ 1.0       ┆ 1.0       ┆ … ┆ 1.0       ┆ 1.0       ┆ 0.0       ┆ 0.0      │
│ 1.0       ┆ 1.0       ┆ 1.0       ┆ 1.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ 0.0       ┆ 1.0       ┆ 1.0       ┆ 1.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ 1.0       ┆ 1.0       ┆ 1.0       ┆ 1.0       ┆ … ┆ 